# Ch.3 — Matrix Factorization

> **The story.** In **1901** Karl Pearson introduced principal components — compressing datasets into fewer dimensions without losing the signal. By the **1960s**, Singular Value Decomposition (SVD) was formalised: any matrix $R = U \Sigma V^T$. The trouble: SVD requires knowing the _entire_ matrix, but a recommender system's ratings matrix is 93.7% empty. The breakthrough came in **2006** when Simon Funk, competing in the **Netflix Prize**, published "Netflix Update: Try This at Home" — only train on ratings you _can_ see, using SGD one rating at a time. "Funky SVD" rocketed him to third on the leaderboard overnight. **Yehuda Koren** at Yahoo! Research formalised the framework with bias terms and temporal dynamics; the BellKor team won the Netflix Prize with 100+ blended models, regularised matrix factorisation as the backbone. In **2012** Steffen **Rendle** published **BPR**, extending factorization to implicit feedback via pairwise ranking — the foundation of modern "Learn to Rank" systems.
>
> **Where you are in the curriculum.** Chapter three. Collaborative filtering (Ch.2) achieved ~65% HR@10 but is crippled by sparsity — most user pairs share fewer than 5 movies. Matrix factorization solves this by mapping users and items into a shared **latent space** where the dot product approximates a rating. Even if two users never rated the same movie, their vectors can be close — capturing "cerebral sci-fi lover" or "90s comedy fan" without those labels ever being defined. This is your first encounter with _latent representations_ — the same idea powering word embeddings, BERT, and every deep learning model in this track.
>
> **Notation.** $R \in \mathbb{R}^{m \times n}$ — rating matrix ($m$ users, $n$ items); $P \in \mathbb{R}^{m \times k}$, $Q \in \mathbb{R}^{n \times k}$ — user/item factor matrices; $k$ — latent factors; $\hat{r}_{ui} = \mathbf{p}_u^\top \mathbf{q}_i + \mu + b_u + b_i$ — predicted rating; $\lambda$ — L2 regularisation; $e_{ui} = r_{ui} - \hat{r}_{ui}$ — prediction error.

---

## §0 · The Challenge — Where We Are

> **The mission**: Launch **FlixAI** — >85% HR@10 across 5 constraints: (1) ACCURACY >85%, (2) COLD START for new users/items, (3) SCALABILITY <200ms, (4) DIVERSITY beyond blockbusters, (5) EXPLAINABILITY "Because you liked X."

**Progress so far:** Ch.1 popularity → 42% HR@10. Ch.2 item-CF → ~65% HR@10. **Still 20 points short.** Item-CF stores an $O(n^2)$ similarity matrix (2.8M floats for 1,682 movies) and fails whenever user pairs share too few ratings. MF with $k=50$ factors compresses everything into 131k floats — 21× more compact — and fills in missing entries through learned structure. The key insight: MF discovers **latent "genre-taste" dimensions** automatically — the dot product $\mathbf{p}_u^\top \mathbf{q}_i$ becomes a taste-to-attribute alignment score.

```mermaid
flowchart LR
 CF["Ch.2: Item-CF\nHR@10 ≈ 65%\nO(n²) similarity"] --> DECOMP["Learn P ∈ ℝ^(m×k)\nQ ∈ ℝ^(n×k)\nvia SGD"]
 DECOMP --> PRED["Predict:\nr̂ = pᵤᵀqᵢ"]
 PRED --> BIAS["+ Bias terms:\nμ + bᵤ + bᵢ"]
 BIAS --> EVAL["MF + bias\nHR@10 ≈ 78%"]

 style CF fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style DECOMP fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style EVAL fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Dataset:** MovieLens 100k | **Task:** SGD matrix factorization with bias terms | **Outcome:** MF = ~78% HR@10


## §1 · The Core Idea

Every user and every movie can be described by a small set of hidden ("latent") preferences. A sci-fi fan has a high score on the "cerebral sci-fi" dimension; _Blade Runner_ has a high score on that same dimension. **You never define these dimensions** — the model discovers them by finding the best $k$-dimensional vectors that reconstruct observed ratings. The dot product between a user vector and an item vector then measures how "aligned" their tastes and attributes are.

```
     R       ≈  P    ·   Qᵀ
  ─────────    ──────   ──────
  [m × n]     [m × k]  [k × n]

  R:  943 users × 1,682 items  (93.7% empty — 100k of 1.6M cells filled)
  P:  943 users   × k latent factors  (learned)
  Q:  1,682 items × k latent factors  (learned)
  k = 20  →  only (943+1682)×20 = 52,500 numbers replace 1.6M
```

The model only trains on the 6.3% of cells that have real ratings. The latent structure fills in the rest — giving a meaningful score for every user-item pair, including those that share zero ratings.

> **Optional depth:** $\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i$ — global mean $\mu$ (e.g. 3.5 stars) plus user bias $b_u$ ("User 42 rates everything 0.3 above average") plus item bias $b_i$ ("Titanic averages 0.5 above global mean") plus the latent alignment $\mathbf{p}_u^\top \mathbf{q}_i$.

**Training objective** — minimise reconstruction error on observed ratings, penalise large vectors (L2 regularisation prevents overfitting on the 6.3% of known ratings):

> **Optional depth:** $\mathcal{L} = \sum_{(u,i) \in \text{obs}} (r_{ui} - \hat{r}_{ui})^2 + \lambda(\|\mathbf{p}_u\|^2 + \|\mathbf{q}_i\|^2 + b_u^2 + b_i^2)$


```mermaid
flowchart LR
    USER["User u\n(943 users)"] --> PVEC["p_u ∈ ℝᵏ\nUser latent vector\n(learned by SGD)"]
    ITEM["Item i\n(1,682 items)"] --> QVEC["q_i ∈ ℝᵏ\nItem latent vector\n(learned by SGD)"]
    PVEC --> DOT["Dot product\np_u · q_i\n+ μ + b_u + b_i"]
    QVEC --> DOT
    DOT --> PRED["r̂_ui\nPredicted rating\n(any user-item pair)"]

    style USER fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style ITEM fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style PVEC fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style QVEC fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style DOT fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style PRED fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

_Visual takeaway: every user-item score is a single dot product — $O(k)$ regardless of how sparse the ratings matrix is. This is why MF handles the sparsity that defeated item-CF._


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)

print("Libraries loaded.")

In [ ]:
# ── Load MovieLens 100k & Split ───────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"]
)

n_users = ratings["user_id"].max()
n_items = ratings["item_id"].max()

# Leave-one-out split
ratings_sorted = ratings.sort_values("timestamp")
test = ratings_sorted.groupby("user_id").tail(1).copy()
train = ratings_sorted.drop(test.index).copy()

print(f"Users: {n_users}  Items: {n_items}")
print(f"Train: {len(train):,}  Test: {len(test):,}")

In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────────────
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    hits = 0
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            hits += 1
    return hits / len(test_df)


def ndcg_at_k(test_df, top_k_per_user, k=10):
    ndcgs = []
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            rank = recs.index(test_item) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)


print("Evaluation functions defined.")

## §2 · SGD-Based Matrix Factorization

Collaborative filtering computed similarity once and looked up neighbours at query time. Matrix factorization learns _continuously_ — each rating in the training set is a teaching signal that nudges the user and item vectors closer together (or further apart, for bad predictions). The SGD update is a hill-climb in vector space: find the prediction error, then adjust each vector in the direction that reduces it, while pulling vectors toward zero (regularisation) to prevent overfitting on the sparse 6.3% of known ratings.

For each observed rating $(u, i, r_{ui})$:

1. Predict: $\hat{r} = \mu + b_u + b_i + \mathbf{p}_u^T \mathbf{q}_i$
2. Error: $e = r_{ui} - \hat{r}$
3. Update:
   - $\mathbf{p}_u \leftarrow \mathbf{p}_u + \eta(e \cdot \mathbf{q}_i - \lambda \cdot \mathbf{p}_u)$
   - $\mathbf{q}_i \leftarrow \mathbf{q}_i + \eta(e \cdot \mathbf{p}_u - \lambda \cdot \mathbf{q}_i)$
   - $b_u \leftarrow b_u + \eta(e - \lambda \cdot b_u)$
   - $b_i \leftarrow b_i + \eta(e - \lambda \cdot b_i)$

> **Optional depth:** The update rule is gradient descent on $\mathcal{L}$. Taking $\partial\mathcal{L}/\partial\mathbf{p}_u = -2e\mathbf{q}_i + 2\lambda\mathbf{p}_u$ and flipping sign gives the ascent direction above. $\eta$ (learning rate) controls step size; too large and the loss oscillates, too small and convergence is slow — typical values: $\eta = 0.005$, $\lambda = 0.02$.


In [ ]:
# ── Matrix Factorization (from scratch) ───────────────────────────────────
class MatrixFactorization:
    def __init__(self, n_users, n_items, n_factors=20, lr=0.005, reg=0.02):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg

        # Initialise latent factors
        self.U = np.random.normal(0, 0.01, (n_users + 1, n_factors))
        self.V = np.random.normal(0, 0.01, (n_items + 1, n_factors))
        self.b_u = np.zeros(n_users + 1)
        self.b_i = np.zeros(n_items + 1)
        self.mu = 0.0

    def fit(self, train_df, n_epochs=50, verbose=True):
        """Train via SGD on observed ratings."""
        self.mu = train_df["rating"].mean()

        users = train_df["user_id"].values
        items = train_df["item_id"].values
        rats = train_df["rating"].values.astype(np.float64)

        train_losses = []
        for epoch in range(n_epochs):
            # Shuffle
            idx = np.random.permutation(len(users))
            epoch_loss = 0.0

            for k in idx:
                u, i, r = users[k], items[k], rats[k]
                pred = self.mu + self.b_u[u] + self.b_i[i] + self.U[u] @ self.V[i]
                err = r - pred
                epoch_loss += err**2

                # Update
                self.U[u] += self.lr * (err * self.V[i] - self.reg * self.U[u])
                self.V[i] += self.lr * (err * self.U[u] - self.reg * self.V[i])
                self.b_u[u] += self.lr * (err - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (err - self.reg * self.b_i[i])

            rmse = np.sqrt(epoch_loss / len(users))
            train_losses.append(rmse)
            if verbose and (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1:>3d}: RMSE = {rmse:.4f}")

        return train_losses

    def predict(self, user_id, item_id):
        return (
            self.mu
            + self.b_u[user_id]
            + self.b_i[item_id]
            + self.U[user_id] @ self.V[item_id]
        )

    def recommend(self, user_id, rated_items, top_k=10):
        scores = self.mu + self.b_u[user_id] + self.b_i + self.U[user_id] @ self.V.T
        scores[list(rated_items)] = -np.inf
        scores[0] = -np.inf  # index 0 unused
        return np.argsort(scores)[-top_k:][::-1].tolist()


print("MatrixFactorization class defined.")

In [ ]:
# ── Train the Model ───────────────────────────────────────────────────────
mf = MatrixFactorization(
    n_users=n_users, n_items=n_items, n_factors=20, lr=0.005, reg=0.02
)
print("Training MF (20 factors, 50 epochs)...")
losses = mf.fit(train, n_epochs=50, verbose=True)

In [ ]:
# ── Training Loss Curve ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(losses) + 1), losses, color="#2980b9", linewidth=2)
ax.set(
    xlabel="Epoch",
    ylabel="Training RMSE",
    title="Matrix Factorization — Training Convergence",
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/mf_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────
user_rated_train = train.groupby("user_id")["item_id"].apply(set).to_dict()

top_k_mf = {}
for user_id in test["user_id"].unique():
    rated = user_rated_train.get(user_id, set())
    recs = mf.recommend(user_id, rated, top_k=10)
    top_k_mf[user_id] = recs

hr_mf = hit_rate_at_k(test, top_k_mf, k=10)
ndcg_mf = ndcg_at_k(test, top_k_mf, k=10)

print(f"Matrix Factorization (d=20):")
print(f"  HR@10   = {hr_mf:.3f} ({hr_mf*100:.1f}%)")
print(f"  NDCG@10 = {ndcg_mf:.4f}")

### What §1–§2 established — and what it still doesn't solve

[Done] **Sparsity solved.** The latent space provides a similarity score for every user-item pair — even those that share zero rated movies.

[Done] **Scalability improved.** Prediction is $O(k)$ — a single dot product — versus $O(n)$ neighbourhood lookups in item-CF.

[Done] **HR@10 lifted.** From ~65% (Ch.2) to ~78% — a 13-point gain from learned structure alone.

**Still open:**

- **Cold start:** A new user has no vector in $P$. The model cannot recommend anything until it has seen at least one rating from that user.
- **Linearity ceiling:** The dot product $\mathbf{p}_u^\top \mathbf{q}_i$ is fundamentally linear — it cannot encode "likes A and B separately but hates A+B together." Ch.4 breaks this ceiling with a neural network.
- **Explainability:** Latent dimensions are discovered, not labelled. The model knows the pattern but cannot say "because you liked sci-fi."


In [ ]:
# ── Effect of Latent Factors (d) ──────────────────────────────────────────
factor_values = [5, 10, 20, 50, 100]
hr_by_d = []

for d in factor_values:
    print(f"  Training MF with d={d}...")
    mf_d = MatrixFactorization(n_users, n_items, n_factors=d, lr=0.005, reg=0.02)
    mf_d.fit(train, n_epochs=30, verbose=False)

    recs_d = {}
    for uid in test["user_id"].unique():
        rated = user_rated_train.get(uid, set())
        recs_d[uid] = mf_d.recommend(uid, rated, top_k=10)
    hr_by_d.append(hit_rate_at_k(test, recs_d, k=10))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    factor_values,
    [h * 100 for h in hr_by_d],
    "o-",
    color="#27ae60",
    linewidth=2,
    markersize=8,
)
ax.set(
    xlabel="Number of Latent Factors (d)",
    ylabel="Hit Rate@10 (%)",
    title="Matrix Factorization: Effect of Latent Dimension",
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/mf_latent_factors.png", dpi=150, bbox_inches="tight")
plt.show()

for d, hr in zip(factor_values, hr_by_d):
    print(f"  d={d:>3d}: HR@10 = {hr:.3f} ({hr*100:.1f}%)")

In [ ]:
# ── Visualise Latent Factors (2D PCA) ─────────────────────────────────────
from sklearn.decomposition import PCA

# Project item factors to 2D
pca = PCA(n_components=2, random_state=SEED)
V_2d = pca.fit_transform(mf.V[1:])  # skip index 0

# Load genre info for colouring
movies_df = pd.read_csv(
    url + "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    names=[
        "item_id",
        "title",
        "release_date",
        "video_release",
        "url",
        "unknown",
        "Action",
        "Adventure",
        "Animation",
        "Children",
        "Comedy",
        "Crime",
        "Documentary",
        "Drama",
        "Fantasy",
        "Film-Noir",
        "Horror",
        "Musical",
        "Mystery",
        "Romance",
        "Sci-Fi",
        "Thriller",
        "War",
        "Western",
    ],
    usecols=range(24),
)

genre_cols = ["Action", "Comedy", "Drama", "Horror", "Sci-Fi", "Romance"]
dominant_genre = movies_df[genre_cols].idxmax(axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
for genre in genre_cols:
    mask = dominant_genre == genre
    mask_idx = movies_df[mask].index
    if len(mask_idx) > 0:
        ax.scatter(V_2d[mask_idx, 0], V_2d[mask_idx, 1], alpha=0.4, s=15, label=genre)

ax.set(
    xlabel="PC 1",
    ylabel="PC 2",
    title="Item Latent Factors — PCA Projection (Coloured by Dominant Genre)",
)
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig("img/mf_latent_space.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# ── Regularisation Effect ─────────────────────────────────────────────────
reg_values = [0.0, 0.001, 0.01, 0.02, 0.05, 0.1, 0.5]
hr_by_reg = []

for reg in reg_values:
    mf_r = MatrixFactorization(n_users, n_items, n_factors=20, lr=0.005, reg=reg)
    mf_r.fit(train, n_epochs=30, verbose=False)

    recs_r = {}
    for uid in test["user_id"].unique():
        rated = user_rated_train.get(uid, set())
        recs_r[uid] = mf_r.recommend(uid, rated, top_k=10)
    hr_by_reg.append(hit_rate_at_k(test, recs_r, k=10))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    reg_values,
    [h * 100 for h in hr_by_reg],
    "o-",
    color="#e74c3c",
    linewidth=2,
    markersize=8,
)
ax.set(
    xlabel="Regularisation λ",
    ylabel="Hit Rate@10 (%)",
    title="Matrix Factorization: Effect of Regularisation",
    xscale="log",
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/mf_regularisation.png", dpi=150, bbox_inches="tight")
plt.show()

for reg, hr in zip(reg_values, hr_by_reg):
    print(f"  λ={reg:.3f}: HR@10 = {hr:.3f} ({hr*100:.1f}%)")

## Progress Check

**Checkpoint:** FlixAI — hit@10 advanced from ~65% toward the >85% target in this chapter. Matrix factorization's latent representations solve the sparsity problem that blocked collaborative filtering.

| #   | Constraint     | Target                | Ch.3 Status                                 |
| --- | -------------- | --------------------- | ------------------------------------------- |
| 1   | ACCURACY       | >85% HR@10            | ~78% (+13 pts from CF)                      |
| 2   | COLD START     | New users/items       | [No] New user = no trained vector           |
| 3   | SCALABILITY    | 1M+ ratings           | SGD scales well — $O(k)$ per step           |
| 4   | DIVERSITY      | Not just popular      | Latent space surfaces niche items           |
| 5   | EXPLAINABILITY | "Because you liked X" | Latent factors are discovered, not labelled |

**Bottom line**: 78% hit rate — latent factors handle sparsity. But the dot product is linear and cannot capture complex taste interactions.

**Next**: Ch.4 — Neural Collaborative Filtering → replace the dot product with a neural network.


## Exercises

**Exercise 1 — Bias Ablation**
Train MF without bias terms ($b_u = b_i = 0$). Compare RMSE and HR@10 against the full model with biases. How much do biases contribute?

**Exercise 2 — ALS Implementation**
Implement Alternating Least Squares: fix V and solve for each $\mathbf{u}_u$ in closed form, then fix U and solve for each $\mathbf{v}_i$. Compare convergence speed against SGD.

**Exercise 3 — Surprise Library**
Use the `surprise` library's SVD implementation. Compare HR@10 against your from-scratch MF. Are there differences?


In [ ]:
# ── Exercise 1 scaffold — Bias Ablation ──────────────────────────────────
# TODO: Modify MatrixFactorization to skip bias terms
# Compare RMSE and HR@10

# class MFNoBias(MatrixFactorization):
#     def fit(self, ...):
#         # Same as parent but don't update b_u, b_i
#         # Predict: r̂ = mu + U[u] @ V[i]  (no bias terms)
#         pass

pass

In [ ]:
# ── Exercise 2 scaffold — ALS ────────────────────────────────────────────
# TODO: Implement Alternating Least Squares
# Fix V, solve for U: u_u = (V_u^T V_u + λI)^{-1} V_u^T r_u
# Fix U, solve for V: v_i = (U_i^T U_i + λI)^{-1} U_i^T r_i

# class ALS:
#     def fit(self, R_sparse, n_epochs=20):
#         for epoch in range(n_epochs):
#             # Fix V, solve for each u
#             # Fix U, solve for each v
#             pass

pass

In [ ]:
# ── Exercise 3 scaffold — Surprise Library ───────────────────────────────
# TODO: pip install scikit-surprise, then:

# from surprise import Dataset, Reader, SVD
# from surprise.model_selection import cross_validate
#
# reader = Reader(rating_scale=(1, 5))
# data = Dataset.load_from_df(ratings[["user_id", "item_id", "rating"]], reader)
# algo = SVD(n_factors=20, lr_all=0.005, reg_all=0.02, n_epochs=50)
# cross_validate(algo, data, measures=["RMSE"], cv=5, verbose=True)

pass